Day 3 — Convolutional Layers From Scratch
Brain Tumour Detection Project
=====================================
Topics covered:
  1.  What Conv2d actually stores — weight tensor anatomy
  2.  Weight sharing — why conv beats fully connected
  3.  Naive convolution from scratch — the four loops
  4.  Stride, padding and the output size formula
  5.  im2col — convolution as one matrix multiply
  6.  Verification against nn.Conv2d — bit-for-bit agreement
  7.  Conv2dScratch — wrapping it as a real nn.Module
  8.  Gradient check — does our layer backprop correctly
  9.  1x1 and grouped convolution — cheap channel mixing
  10. ConvBlock assembly + verification checklist

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

import os
os.makedirs("outputs", exist_ok=True)
torch.manual_seed(42)

In [ ]:
# 1. WHAT Conv2d ACTUALLY STORES — WEIGHT TENSOR ANATOMY
"""
nn.Conv2d(in_ch, out_ch, K) holds exactly two tensors:

  weight : (out_ch, in_ch, K, K)
  bias   : (out_ch,)            — or None when bias=False

Read the weight shape right-to-left:
  (K, K)            one 2D kernel
  (in_ch, K, K)     one FILTER — it spans ALL input channels
  (out_ch, ...)     out_ch independent filters stacked

The key point that trips people up:
  a filter is NOT 2D. To produce ONE output channel you slide a
  3D block of shape (in_ch, K, K) over the image, multiply
  element-wise, and sum EVERYTHING — across space AND channels.
  That single sum is one number in the output feature map.

  → output channel c depends on every input channel
  → in_ch collapses to 1 per filter; out_ch is how many you run
"""
conv = nn.Conv2d(in_channels=3, out_channels=8, kernel_size=3, bias=True)

print(f"nn.Conv2d(3, 8, 3)")
print(f"  weight shape : {tuple(conv.weight.shape)}   (out_ch, in_ch, K, K)")
print(f"  bias   shape : {tuple(conv.bias.shape)}")
print(f"\n  one filter   : {tuple(conv.weight[0].shape)}  -> spans all 3 input channels")
print(f"  one kernel   : {tuple(conv.weight[0, 0].shape)}  -> a single 2D slice")

n_w = conv.weight.numel()
n_b = conv.bias.numel()
print(f"\n  weight params: 8 x 3 x 3 x 3 = {n_w}")
print(f"  bias   params: 8            = {n_b}")
print(f"  TOTAL        : {n_w + n_b}")

print(f"\nFor our grayscale MRI first layer, nn.Conv2d(1, 32, 3, bias=False):")
c0 = nn.Conv2d(1, 32, 3, bias=False)
print(f"  weight shape : {tuple(c0.weight.shape)}")
print(f"  params       : 32 x 1 x 3 x 3 = {c0.weight.numel()}")
print(f"  bias         : None — BatchNorm's beta does the shifting instead")

In [ ]:
# 2. WEIGHT SHARING — WHY CONV BEATS FULLY CONNECTED
"""
Weight sharing = the SAME kernel is reused at every spatial position.

A fully connected layer learns a separate weight for every
(input pixel, output pixel) pair. A conv layer learns ONE small
kernel and slides it everywhere.

Two consequences, both essential for medical imaging:

  1. Parameter count collapses.
     FC needs (H x W x in_ch) x (H x W x out_ch) weights.
     Conv needs out_ch x in_ch x K x K — independent of image size.

  2. Translation equivariance.
     A tumour in the top-left and the same tumour in the bottom-right
     are detected by the SAME weights. An FC layer would have to learn
     the pattern separately at every location, and would therefore need
     to SEE it at every location during training. We have ~1000 scans;
     that is hopeless.

This is the entire reason CNNs work on images and MLPs do not.
"""
H = W = 128
in_ch, out_ch, K = 1, 32, 3

fc_params   = (H * W * in_ch) * (H * W * out_ch)
conv_params = out_ch * in_ch * K * K

print(f"Input {in_ch}x{H}x{W}  ->  output {out_ch}x{H}x{W}")
print(f"  Fully connected : {fc_params:>18,} weights")
print(f"  Convolution     : {conv_params:>18,} weights")
print(f"  Ratio           : {fc_params / conv_params:>18,.0f}x fewer")

print(f"\n  FC at float32   : {fc_params * 4 / 1e9:.1f} GB for ONE layer")
print(f"  Conv at float32 : {conv_params * 4:.0f} bytes")

print(f"\nConv params do NOT grow with image size:")
print(f"  {'image':>10} {'FC weights':>20} {'Conv weights':>14}")
for size in (32, 64, 128, 256):
    fc = (size * size * in_ch) * (size * size * out_ch)
    print(f"  {size:>7}px {fc:>20,} {conv_params:>14,}")
print(f"  -> conv is constant; FC grows with the FOURTH power of size")

sizes = np.array([32, 64, 128, 256])
fc_counts = (sizes**2 * in_ch) * (sizes**2 * out_ch)
conv_counts = np.full_like(sizes, conv_params)

fig, ax = plt.subplots(figsize=(7, 4))
fig.patch.set_facecolor('#F8F8F6')
ax.plot(sizes, fc_counts, 'o-', color='#D85A30', lw=2.5, label='Fully connected')
ax.plot(sizes, conv_counts, 'o-', color='#1D9E75', lw=2.5, label='Convolution (3x3)')
ax.set_yscale('log')
ax.set_xlabel("Image size (px)")
ax.set_ylabel("Weights in one layer (log scale)")
ax.set_title("Weight sharing: conv params are independent of image size",
             fontsize=10, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("outputs/weight_sharing.png", dpi=120, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.close(fig)
print("\n  saved -> outputs/weight_sharing.png")

In [ ]:
# 3. NAIVE CONVOLUTION FROM SCRATCH — THE FOUR LOOPS
"""
The literal definition, written out as nested loops:

  for each image n
    for each output channel co
      for each output row i
        for each output col j
          out[n,co,i,j] = sum over (ci, kh, kw) of
                            x[n, ci, i*stride+kh, j*stride+kw]
                          x w[co, ci, kh, kw]

This is correct but slow — it is Python-level looping over every
output position. We use a tiny 8x8 input here for that reason.
Sections 5-6 replace it with one matrix multiply.

Note there is no flipping of the kernel. What deep learning calls
'convolution' is really CROSS-CORRELATION. True convolution flips
the kernel first; since the kernel is learned, the flip is
irrelevant and every framework omits it.
"""
def conv2d_naive(x, weight, bias=None, stride=1, padding=0):
    """Reference implementation — clarity over speed."""
    N, C_in, H, W = x.shape
    C_out, _, KH, KW = weight.shape

    if padding > 0:
        x = F.pad(x, (padding, padding, padding, padding))

    H_out = (H + 2 * padding - KH) // stride + 1
    W_out = (W + 2 * padding - KW) // stride + 1
    out = torch.zeros(N, C_out, H_out, W_out, dtype=x.dtype)

    for n in range(N):
        for co in range(C_out):
            for i in range(H_out):
                for j in range(W_out):
                    hs, ws = i * stride, j * stride
                    patch = x[n, :, hs:hs + KH, ws:ws + KW]   # (C_in, KH, KW)
                    out[n, co, i, j] = (patch * weight[co]).sum()
            if bias is not None:
                out[n, co] += bias[co]
    return out


x_small = torch.randn(1, 2, 8, 8)
w_small = torch.randn(3, 2, 3, 3)
b_small = torch.randn(3)

mine = conv2d_naive(x_small, w_small, b_small, stride=1, padding=1)
ref  = F.conv2d(x_small, w_small, b_small, stride=1, padding=1)

print(f"input  {tuple(x_small.shape)}  weight {tuple(w_small.shape)}")
print(f"naive  -> {tuple(mine.shape)}")
print(f"F.conv2d -> {tuple(ref.shape)}")
print(f"\nmax abs difference: {(mine - ref).abs().max().item():.3e}")
print(f"allclose(atol=1e-6): {torch.allclose(mine, ref, atol=1e-6)}")

In [ ]:
# 4. STRIDE, PADDING AND THE OUTPUT SIZE FORMULA
"""
  H_out = floor((H_in + 2*padding - K) / stride) + 1

padding:
  'same' padding keeps H_out == H_in when stride=1.
  For odd K that means padding = (K-1)//2  -> K=3 needs padding=1.
  Without it every conv shrinks the map by 2px, and after 4 blocks
  a 128px image would silently lose 8px of border.

stride:
  stride=1 looks at every position (we use this).
  stride=2 skips every other position, halving the output —
  an alternative to pooling. We use MaxPool instead (Day 4)
  so that downsampling stays separate from feature extraction.

Why bias=False in our ConvBlock:
  BatchNorm immediately subtracts the batch mean, which cancels any
  constant the bias added. The bias is not wrong, just redundant —
  and it costs out_ch parameters per layer for nothing.
"""
print(f"{'H_in':>6} {'K':>3} {'pad':>4} {'stride':>7} {'H_out':>7}  {'note'}")
print("-" * 58)
for H_in, K, pad, s, note in [
    (128, 3, 0, 1, "no padding -> shrinks by K-1 = 2"),
    (128, 3, 1, 1, "'same' padding -> size preserved"),
    (128, 3, 1, 2, "stride 2 -> halves the map"),
    (128, 5, 2, 1, "K=5 needs pad=2 for 'same'"),
    (128, 7, 3, 1, "K=7 needs pad=3 for 'same'"),
]:
    H_out = (H_in + 2 * pad - K) // s + 1
    print(f"{H_in:>6} {K:>3} {pad:>4} {s:>7} {H_out:>7}  {note}")

print(f"\nVerifying 'same' padding = (K-1)//2 for odd kernels:")
for K in (1, 3, 5, 7, 9):
    pad = (K - 1) // 2
    probe = F.conv2d(torch.zeros(1, 1, 128, 128),
                     torch.zeros(1, 1, K, K), padding=pad)
    ok = probe.shape[-1] == 128
    print(f"  K={K}  pad={pad}  ->  {probe.shape[-1]}px  {'OK' if ok else 'FAIL'}")

print(f"\nWhat happens WITHOUT padding, over 4 blocks:")
size = 128
for block in range(1, 5):
    size = size - 2
    print(f"  after block {block}: {size}px  (lost {128 - size}px of border)")

In [ ]:
# 5. im2col — CONVOLUTION AS ONE MATRIX MULTIPLY
"""
The trick that makes convolution fast on a GPU.

Every output position reads a (C_in, K, K) patch. im2col flattens
each patch into a column, stacking them into one big matrix:

  cols   : (N, C_in*K*K, L)     L = H_out * W_out  patches
  w_flat : (C_out, C_in*K*K)    one filter per row

  out = w_flat @ cols           -> (N, C_out, L)  -> reshape

Now the whole layer is a single dense matrix multiply, which is the
one operation GPUs are overwhelmingly optimised for.

The cost: patches OVERLAP, so cols duplicates input values. With
K=3, stride=1 each pixel appears in up to 9 columns — roughly 9x
the memory of the input. Speed is bought with memory.

torch.nn.functional.unfold IS im2col. We use it rather than
rebuilding the indexing by hand, because it is differentiable —
which is what makes section 7 work.
"""
def conv2d_im2col(x, weight, bias=None, stride=1, padding=0):
    """Same result as conv2d_naive, but one matmul instead of four loops."""
    N, C_in, H, W = x.shape
    C_out, _, KH, KW = weight.shape

    H_out = (H + 2 * padding - KH) // stride + 1
    W_out = (W + 2 * padding - KW) // stride + 1

    cols = F.unfold(x, (KH, KW), padding=padding, stride=stride)
    w_flat = weight.view(C_out, -1)

    out = w_flat @ cols                       # (N, C_out, L) via broadcasting
    if bias is not None:
        out = out + bias.view(1, -1, 1)
    return out.view(N, C_out, H_out, W_out)


x_d = torch.randn(1, 2, 8, 8)
cols = F.unfold(x_d, (3, 3), padding=1, stride=1)
print(f"input          : {tuple(x_d.shape)}")
print(f"after unfold   : {tuple(cols.shape)}   (N, C_in*K*K, L)")
print(f"  C_in*K*K = 2 x 3 x 3 = {2*3*3}")
print(f"  L        = 8 x 8     = {8*8}  output positions")

w_flat = w_small.view(3, -1)
print(f"weight flat    : {tuple(w_flat.shape)}   (C_out, C_in*K*K)")
print(f"  matmul       : ({w_flat.shape[0]}, {w_flat.shape[1]}) @ "
      f"({cols.shape[1]}, {cols.shape[2]}) -> (3, 64) -> (1, 3, 8, 8)")

mem_in = x_d.numel()
print(f"\nmemory blow-up : {cols.numel()} values vs {mem_in} in the input "
      f"({cols.numel()/mem_in:.1f}x)")

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
fig.patch.set_facecolor('#F8F8F6')
axes[0].imshow(x_d[0, 0].numpy(), cmap='gray')
axes[0].set_title("Input feature map\n(1 channel of 2, 8x8)",
                  fontsize=10, fontweight='bold')
axes[0].axis('off')
im = axes[1].imshow(cols[0].numpy(), cmap='viridis', aspect='auto')
axes[1].set_title("After im2col / unfold\n(18 x 64 — each column is one patch)",
                  fontsize=10, fontweight='bold')
axes[1].set_xlabel("output position (L)")
axes[1].set_ylabel("C_in x K x K")
fig.colorbar(im, ax=axes[1], fraction=0.046)
plt.suptitle("Convolution rewritten as a matrix multiply",
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig("outputs/im2col.png", dpi=120, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.close(fig)
print("  saved -> outputs/im2col.png")

In [ ]:
# 6. VERIFICATION AGAINST nn.Conv2d — BIT-FOR-BIT AGREEMENT
"""
An implementation is only trustworthy if it matches the reference
across the whole parameter space, not on one lucky example.

We sweep stride, padding, kernel size, channel counts and batch
size, and compare BOTH implementations against F.conv2d.

Tolerance note — this matters more than it looks.
  float32 holds ~7 significant digits, and each output value is a
  sum of C_in x K x K products. A 7x7 kernel over 4 channels sums
  196 terms, so rounding error accumulates and the ABSOLUTE
  difference grows with kernel size. Comparing against a fixed
  absolute threshold would wrongly flag big kernels as broken.

  So we compare RELATIVE error: |mine - ref| / max|ref|.
  That stays flat across kernel sizes, because both the error and
  the magnitude scale together.
"""
cases = [
    # (N, C_in, C_out, K, stride, padding, H)
    (1, 1,  4,  3, 1, 1, 16),
    (2, 3,  8,  3, 1, 1, 16),
    (2, 3,  8,  3, 2, 1, 16),
    (1, 2,  5,  5, 1, 2, 16),
    (1, 2,  5,  5, 2, 0, 16),
    (4, 1, 32,  3, 1, 1, 12),
    (1, 8, 16,  1, 1, 0, 12),
    (2, 4,  6,  7, 1, 3, 16),
]

RTOL = 1e-5

print(f"{'N':>2} {'Cin':>4} {'Cout':>5} {'K':>2} {'s':>2} {'p':>2} {'H':>4} "
      f"{'terms':>6} {'out shape':>18} {'naive rel':>11} {'im2col rel':>11}")
print("-" * 89)

all_ok = True
for N, C_in, C_out, K, s, p, H in cases:
    x = torch.randn(N, C_in, H, H)
    w = torch.randn(C_out, C_in, K, K)
    b = torch.randn(C_out)

    ref = F.conv2d(x, w, b, stride=s, padding=p)
    scale = ref.abs().max().item()          # normalise by output magnitude

    e_naive = (conv2d_naive(x, w, b, s, p) - ref).abs().max().item() / scale
    e_col   = (conv2d_im2col(x, w, b, s, p) - ref).abs().max().item() / scale

    ok = max(e_naive, e_col) < RTOL
    all_ok &= ok
    print(f"{N:>2} {C_in:>4} {C_out:>5} {K:>2} {s:>2} {p:>2} {H:>4} "
          f"{C_in*K*K:>6} {str(tuple(ref.shape)):>18} "
          f"{e_naive:>11.2e} {e_col:>11.2e}{'' if ok else '  <-- FAIL'}")

print("-" * 89)
print(f"All {len(cases)} cases match F.conv2d (rtol={RTOL:.0e}): {all_ok}")
print(f"\n  'terms' = C_in x K x K values summed per output.")
print(f"  More terms -> more float32 rounding, which is why the")
print(f"  comparison is relative rather than absolute.")

In [ ]:
# 7. Conv2dScratch — WRAPPING IT AS A REAL nn.Module
"""
A working forward pass is not yet a layer. To drop into a model it
must be an nn.Module that:

  - registers weight/bias as nn.Parameter, so .parameters() finds
    them and the optimiser can update them
  - initialises sensibly (Kaiming, matching Day 1)
  - moves with .to(device), .train(), .eval() for free

We do NOT write a backward(). Because conv2d_im2col is built from
differentiable ops (unfold, matmul, add), autograd derives the
backward pass automatically. That is the payoff for expressing
convolution in terms of existing tensor operations.

Kaiming (He) init, mode='fan_in':
  std = sqrt(2 / (in_ch * K * K))
  The 2 compensates for ReLU zeroing half the activations. Without
  it, signal variance shrinks layer by layer and deep stacks stop
  learning — demonstrated on Day 1.
"""
class Conv2dScratch(nn.Module):
    def __init__(self, in_ch: int, out_ch: int, kernel_size: int = 3,
                 stride: int = 1, padding: int = 0, bias: bool = True):
        super().__init__()
        self.in_ch, self.out_ch = in_ch, out_ch
        self.kernel_size, self.stride, self.padding = kernel_size, stride, padding

        self.weight = nn.Parameter(
            torch.empty(out_ch, in_ch, kernel_size, kernel_size))
        nn.init.kaiming_normal_(self.weight, mode='fan_in', nonlinearity='relu')

        if bias:
            self.bias = nn.Parameter(torch.zeros(out_ch))
        else:
            self.register_parameter('bias', None)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return conv2d_im2col(x, self.weight, self.bias,
                             self.stride, self.padding)

    def extra_repr(self) -> str:
        return (f"{self.in_ch}, {self.out_ch}, kernel_size={self.kernel_size}, "
                f"stride={self.stride}, padding={self.padding}, "
                f"bias={self.bias is not None}")


layer = Conv2dScratch(1, 32, 3, padding=1, bias=False)
x = torch.randn(4, 1, 128, 128)
y = layer(x)

print(f"{layer}")
print(f"\n  input  : {tuple(x.shape)}")
print(f"  output : {tuple(y.shape)}")
print(f"  params : {sum(p.numel() for p in layer.parameters()):,}")

fan_in = 1 * 3 * 3
print(f"\nKaiming init check:")
print(f"  expected std = sqrt(2/{fan_in}) = {np.sqrt(2/fan_in):.4f}")
print(f"  actual   std = {layer.weight.std().item():.4f}")

print(f"\nRegistered as real parameters:")
for name, p in layer.named_parameters():
    print(f"  {name:<8} {str(tuple(p.shape)):>18}  requires_grad={p.requires_grad}")

In [ ]:
# 8. GRADIENT CHECK — DOES OUR LAYER BACKPROP CORRECTLY
"""
Forward agreement does not prove backward agreement. A layer that
predicts correctly but computes wrong gradients will train to
garbage, and the failure is silent.

Test: give our layer and nn.Conv2d IDENTICAL weights and input,
backprop the same scalar loss, then compare d(loss)/d(weight) and
d(loss)/d(input). If both match, our layer is a drop-in replacement.
"""
torch.manual_seed(0)
x_ref = torch.randn(2, 3, 16, 16, requires_grad=True)
x_our = x_ref.detach().clone().requires_grad_(True)

ref_layer = nn.Conv2d(3, 6, 3, padding=1, bias=True)
our_layer = Conv2dScratch(3, 6, 3, padding=1, bias=True)

# force identical weights
with torch.no_grad():
    our_layer.weight.copy_(ref_layer.weight)
    our_layer.bias.copy_(ref_layer.bias)

out_ref = ref_layer(x_ref)
out_our = our_layer(x_our)

# identical scalar loss so gradients are directly comparable
out_ref.pow(2).sum().backward()
out_our.pow(2).sum().backward()

print(f"forward   max|diff| : {(out_ref - out_our).abs().max().item():.3e}")
print(f"d/d weight max|diff|: "
      f"{(ref_layer.weight.grad - our_layer.weight.grad).abs().max().item():.3e}")
print(f"d/d bias   max|diff|: "
      f"{(ref_layer.bias.grad - our_layer.bias.grad).abs().max().item():.3e}")
print(f"d/d input  max|diff|: "
      f"{(x_ref.grad - x_our.grad).abs().max().item():.3e}")

checks = {
    "forward":  torch.allclose(out_ref, out_our, atol=1e-5),
    "d weight": torch.allclose(ref_layer.weight.grad, our_layer.weight.grad, atol=1e-4),
    "d bias":   torch.allclose(ref_layer.bias.grad, our_layer.bias.grad, atol=1e-4),
    "d input":  torch.allclose(x_ref.grad, x_our.grad, atol=1e-4),
}
print()
for name, ok in checks.items():
    print(f"  {name:<9} {'OK' if ok else 'FAIL'}")
grads_match = all(checks.values())
print(f"\nConv2dScratch is a drop-in replacement: {grads_match}")

print(f"\nDouble-precision gradcheck (the strict test):")
gc_layer = Conv2dScratch(2, 3, 3, padding=1).double()
gc_x = torch.randn(1, 2, 6, 6, dtype=torch.double, requires_grad=True)
passed = torch.autograd.gradcheck(gc_layer, (gc_x,), eps=1e-6, atol=1e-4)
print(f"  torch.autograd.gradcheck: {passed}")

In [ ]:
# 9. 1x1 AND GROUPED CONVOLUTION — CHEAP CHANNEL MIXING
"""
1x1 convolution
  Looks pointless — a 1x1 kernel sees a single pixel. But remember
  a filter spans ALL input channels, so a 1x1 conv is a fully
  connected layer applied independently at every spatial position.
  It mixes CHANNELS while touching no neighbours.

  Used to cheaply change channel depth: 256 -> 64 costs
  64 x 256 x 1 x 1 = 16,384 params instead of 147,456 for 3x3.

Grouped convolution
  Split channels into g groups; each filter only sees its own group.
  Params drop by exactly g. groups=in_ch is a DEPTHWISE conv —
  the basis of MobileNet.

  Trade-off: groups block information flow BETWEEN groups, which is
  why depthwise convs are always paired with a 1x1 to remix.

We use plain dense 3x3 convs in this project: our model is small
enough that the parameter saving is unnecessary, and dense
connectivity gives the optimiser the most freedom.
"""
C_in, C_out = 256, 64
print(f"{'variant':<28} {'params':>10}  {'note'}")
print("-" * 72)
p3 = C_out * C_in * 3 * 3
p1 = C_out * C_in * 1 * 1
print(f"{'3x3 dense':<28} {p3:>10,}  full spatial + channel mixing")
print(f"{'1x1 dense':<28} {p1:>10,}  channel mixing only ({p3/p1:.0f}x cheaper)")

for g in (2, 4, 8):
    pg = (C_out // g) * (C_in // g) * 3 * 3 * g
    print(f"{'3x3 grouped, groups=' + str(g):<28} {pg:>10,}  "
          f"{p3/pg:.0f}x cheaper than dense")

pdw = C_in * 1 * 3 * 3
print(f"{'3x3 depthwise (groups=C_in)':<28} {pdw:>10,}  no channel mixing at all")

print(f"\nVerifying a 1x1 conv is a per-pixel linear layer:")
x_11 = torch.randn(1, 4, 5, 5)
c_11 = nn.Conv2d(4, 2, 1, bias=False)
out_11 = c_11(x_11)

pixel = x_11[0, :, 2, 3]                       # (4,) the channel vector at (2,3)
manual = c_11.weight.view(2, 4) @ pixel        # plain matrix-vector product
print(f"  conv output at (2,3) : {out_11[0, :, 2, 3].tolist()}")
print(f"  manual  W @ pixel    : {manual.tolist()}")
print(f"  match: {torch.allclose(out_11[0, :, 2, 3], manual, atol=1e-6)}")

print(f"\nVerifying grouped conv parameter count:")
for g in (1, 2, 4):
    cg = nn.Conv2d(8, 8, 3, groups=g, bias=False)
    print(f"  groups={g}: weight {tuple(cg.weight.shape)} = "
          f"{cg.weight.numel():>5,} params")

In [ ]:
# 10. ConvBlock ASSEMBLY + VERIFICATION CHECKLIST
"""
The unit Day 4 stacks four times:

  Conv2d(bias=False) -> BatchNorm2d -> ReLU(inplace)

Order matters, and each choice has a reason:

  Conv first     extracts the features.
  BN second      normalises PRE-activation, so ReLU receives a
                 zero-centred unit-variance distribution and roughly
                 half the units fire. BN after ReLU would normalise
                 an already one-sided distribution.
  ReLU last      non-linearity. inplace=True overwrites its input
                 to save memory — safe here because nothing else
                 reads the BN output.

  bias=False     BN subtracts the batch mean, cancelling any
                 constant the conv bias added. Redundant parameters.

'same' padding keeps spatial size inside the block; downsampling is
MaxPool's job, applied BETWEEN blocks (Day 4).

We use nn.Conv2d rather than Conv2dScratch from here on. They are
verified equivalent in sections 6 and 8, but cuDNN's implementation
is far faster than im2col + matmul.
"""
class ConvBlock(nn.Module):
    """
    Conv2d(bias=False) -> BatchNorm2d -> ReLU(inplace)
    Kaiming initialised. Same padding preserves spatial size.
    """
    def __init__(self, in_ch: int, out_ch: int,
                 kernel_size: int = 3, stride: int = 1):
        super().__init__()
        padding      = (kernel_size - 1) // 2
        self.conv    = nn.Conv2d(in_ch, out_ch, kernel_size,
                                 stride=stride, padding=padding, bias=False)
        self.bn      = nn.BatchNorm2d(out_ch)
        self.relu    = nn.ReLU(inplace=True)
        nn.init.kaiming_normal_(self.conv.weight,
                                mode='fan_in', nonlinearity='relu')
        nn.init.ones_(self.bn.weight)
        nn.init.zeros_(self.bn.bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.relu(self.bn(self.conv(x)))


block = ConvBlock(1, 32)
x = torch.randn(8, 1, 128, 128)
block.train()
out = block(x)

print(f"ConvBlock(1, 32)")
print(f"  input  : {tuple(x.shape)}")
print(f"  output : {tuple(out.shape)}   <- spatial size preserved")

conv_p = sum(p.numel() for p in block.conv.parameters())
bn_p   = sum(p.numel() for p in block.bn.parameters())
print(f"\n  conv params : {conv_p:>5,}  (32 x 1 x 3 x 3, no bias)")
print(f"  bn   params : {bn_p:>5,}  (gamma + beta, 32 each)")
print(f"  TOTAL       : {conv_p + bn_p:>5,}")

with torch.no_grad():
    pre_bn  = block.conv(x)
    post_bn = block.bn(pre_bn)
print(f"\nDistribution through the block:")
print(f"  after conv : mean={pre_bn.mean():+.4f}  std={pre_bn.std():.4f}")
print(f"  after BN   : mean={post_bn.mean():+.4f}  std={post_bn.std():.4f}"
      f"   <- normalised")
print(f"  after ReLU : mean={out.mean():+.4f}  std={out.std():.4f}")
print(f"  fraction of activations alive: {(out > 0).float().mean():.3f}"
      f"   <- want ~0.5")

out.pow(2).mean().backward()
gn = block.conv.weight.grad.norm().item()

print("\n" + "=" * 60)
print("DAY 3 VERIFICATION CHECKLIST")
print("=" * 60)
alive = (out > 0).float().mean().item()
final_checks = [
    ("all 8 forward sweep cases match F.conv2d", all_ok),
    ("Conv2dScratch gradients match nn.Conv2d",  grads_match),
    ("gradcheck passes in float64",              passed),
    ("ConvBlock preserves 128x128",              tuple(out.shape) == (8, 32, 128, 128)),
    ("BN output is zero-centred",                abs(post_bn.mean().item()) < 1e-3),
    ("~half of ReLU units are alive",            0.4 < alive < 0.6),
    ("gradients flow to conv weights",           gn > 0),
]
for label, ok in final_checks:
    print(f"  {'OK  ' if ok else 'FAIL'}  {label}")

print(f"\n  ConvBlock is ready — Day 4 stacks four of these with MaxPool.")